# Search database inspection

Read-only browser for the run/task trace database (`search_db.sqlite`) and the base attribute extraction cache (`.cache/base_extraction.sqlite`). Run all cells from top to bottom; every cell is read-only except the explicitly opt-in `base_cache` clearing cell, which is a no-op by default.

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "search").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")


# When launched from src/search/script this matches the module's existing
# notebook bootstrap (two directories up); parent discovery also supports Run All
# commands started at the repository root.
PROJECT_ROOT = find_project_root(Path(os.getcwd()).resolve())
TRACE_DB_PATH = PROJECT_ROOT / "search_db.sqlite"
CACHE_DB_PATH = PROJECT_ROOT / ".cache" / "base_extraction.sqlite"

for path in (TRACE_DB_PATH, CACHE_DB_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Database not found: {path}")

# mode=ro makes SQLite reject all write operations from this notebook.
TRACE_CONN = sqlite3.connect(f"file:{TRACE_DB_PATH}?mode=ro", uri=True)
CACHE_CONN = sqlite3.connect(f"file:{CACHE_DB_PATH}?mode=ro", uri=True)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", None)

print(f"Project root: {PROJECT_ROOT}")
print(f"Trace DB (read-only): {TRACE_DB_PATH}")
print(f"Cache DB (read-only): {CACHE_DB_PATH}")


def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def table_names(conn: sqlite3.Connection) -> list[str]:
    return [
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name"
        )
    ]


def show_query(label: str, query: str, conn: sqlite3.Connection, *, head: int | None = None) -> pd.DataFrame:
    df = pd.read_sql_query(query, conn)
    print(f"{label}: {df.shape[0]} rows × {df.shape[1]} columns")
    display(df.head(head) if head is not None else df)
    return df


## Summary — row counts

In [ ]:
row_counts = []
for database, conn in [(".cache/base_extraction.sqlite", CACHE_CONN), ("search_db.sqlite", TRACE_CONN)]:
    for table in table_names(conn):
        row_count = conn.execute(f"SELECT COUNT(*) FROM {quote_identifier(table)}").fetchone()[0]
        row_counts.append({"database": database, "table": table, "row_count": row_count})

summary_df = pd.DataFrame(row_counts).sort_values(["database", "table"]).reset_index(drop=True)
display(summary_df)

## .cache/base_extraction.sqlite — base_cache

This append-only table persists across runs. The cell below can wipe it when cached titles need re-extraction.

In [ ]:
base_cache_df = show_query("base_cache (first 20)", "SELECT * FROM base_cache", CACHE_CONN, head=20)

In [ ]:
# Clearing is opt-in: set to True and re-run THIS cell only.
# base_cache is append-only, so entries survive across runs; wipe it when
# brand.xlsx or the numeric rules change and cached titles must be re-extracted.
CLEAR_BASE_CACHE = False

if not CLEAR_BASE_CACHE:
    print(
        f"base_cache left untouched ({base_cache_df.shape[0]} rows shown above). "
        "Set CLEAR_BASE_CACHE = True and re-run this cell to delete every row."
    )
else:
    # Separate read-write connection; CACHE_CONN stays mode=ro.
    writer = sqlite3.connect(CACHE_DB_PATH)
    try:
        with writer:
            deleted = writer.execute("DELETE FROM base_cache").rowcount
        remaining = writer.execute("SELECT COUNT(*) FROM base_cache").fetchone()[0]
    finally:
        writer.close()
    print(f"Deleted {deleted} rows from base_cache; {remaining} remain.")
    print("The table is kept (schema is recreated idempotently anyway); "
          "the next pipeline run re-extracts and repopulates it.")

## search_db.sqlite — runs

In [ ]:
runs_df = show_query("runs", "SELECT * FROM runs", TRACE_CONN)

## search_db.sqlite — tasks

In [ ]:
tasks_df = show_query("tasks", "SELECT * FROM tasks", TRACE_CONN)

## search_db.sqlite — attempts

In [ ]:
attempts_df = show_query("attempts", "SELECT * FROM attempts", TRACE_CONN)

## search_db.sqlite — node_events

In [ ]:
node_events_df = show_query("node_events", "SELECT * FROM node_events", TRACE_CONN)

## search_db.sqlite — candidates

In [ ]:
candidates_df = show_query("candidates", "SELECT * FROM candidates", TRACE_CONN)

## search_db.sqlite — llm_calls

In [ ]:
llm_calls_df = show_query("llm_calls", "SELECT * FROM llm_calls", TRACE_CONN)

## search_db.sqlite — meta

In [ ]:
meta_df = show_query("meta", "SELECT * FROM meta", TRACE_CONN)

## search_db.sqlite — views

### v_errors

In [ ]:
v_errors_df = show_query("v_errors", "SELECT * FROM v_errors", TRACE_CONN)

### v_task_result

In [ ]:
v_task_result_df = show_query("v_task_result", "SELECT * FROM v_task_result", TRACE_CONN)

### v_funnel

In [ ]:
v_funnel_df = show_query("v_funnel", "SELECT * FROM v_funnel", TRACE_CONN)

### v_run_summary

In [ ]:
v_run_summary_df = show_query("v_run_summary", "SELECT * FROM v_run_summary", TRACE_CONN)

## Close connections

The cells above use read-only connections, except the clearing cell, which opens and closes its own read-write connection. Run the final cell when you are finished with the notebook.

In [ ]:
TRACE_CONN.close()
CACHE_CONN.close()
print("Read-only database connections closed.")